# 01 – Ingest e harmonização MODIS + VIIRS

Lê `raw/**/<sensor>_<ano>_all_countries.zip` da Drive (cada zip tem `<sensor>/<ano>/<sensor>_<ano>_<País>.csv`),
harmoniza as colunas (ver skill `firms-active-fire-attributes`) e grava **um Parquet por sensor e ano** em `processed/`.

- Processa ano a ano (o VIIRS tem ~3 GB comprimidos) e **salta** o que já existe, para poupar créditos.
- Preserva os valores originais (`confidence_raw`, `version`, `acq_time` como texto, nomes das bandas de brilho).
- Não calcula métricas: isso é o `02_metrics`.

**Antes de correr:** a pasta `Sprinklers` pertence a outra conta. Em drive.google.com clique com o botão direito em
`Sprinklers` → *Organizar → Adicionar atalho* → *O Meu Drive*, ou ajuste `ROOT` abaixo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/Sprinklers')   # ajustar se o atalho tiver outro nome/local
RAW = ROOT / 'raw'
PROCESSED = ROOT / 'processed'

assert RAW.exists(), f"Nao encontro {RAW}. Adicione um atalho de 'Sprinklers' ao 'O Meu Drive' (ver topo)."
PROCESSED.mkdir(exist_ok=True)

# Filtros opcionais para testar com pouco: ex. SENSORS = {'modis'}; YEARS = {2000}
SENSORS = None    # None = todos
YEARS = None      # None = todos
FORCE = False     # True = refaz mesmo que o Parquet ja exista

In [ ]:
import re, zipfile, shutil, tempfile
import numpy as np
import pandas as pd

ZIP_RE = re.compile(r"^(?P<sensor>modis|viirs-[a-z0-9]+)_(?P<year>\d{4})_all_countries\.zip$", re.I)
CSV_RE = re.compile(r"^(?P<sensor>modis|viirs-[a-z0-9]+)_(?P<year>\d{4})_(?P<country>.+)\.csv$", re.I)

NUMERIC = ["latitude", "longitude", "scan", "track", "frp"]


def find_zips(raw):
    """(sensor, ano) -> caminho. Se houver duplicados (mesmo nome), fica o mais recente."""
    found = {}
    for p in sorted(raw.rglob("*.zip")):
        m = ZIP_RE.match(p.name)
        if not m:
            continue
        key = (m["sensor"].lower(), int(m["year"]))
        if key in found:
            print(f"AVISO duplicado {key}: {found[key]} e {p} -> fica o mais recente")
            if p.stat().st_mtime < found[key].stat().st_mtime:
                continue
        found[key] = p
    return found


def confidence_class(conf):
    """MODIS 0-100 -> l/n/h (0-29, 30-79, 80-100); VIIRS ja vem l/n/h."""
    s = conf.astype("string").str.strip().str.lower()
    num = pd.to_numeric(s, errors="coerce").to_numpy(dtype="float64", na_value=np.nan)   # numpy puro (sem NA) p/ np.select
    cls = pd.Series(np.select([num < 30, (num >= 30) & (num < 80), num >= 80], ["l", "n", "h"], default=""),
                    index=s.index, dtype="object")
    first = s.str[:1]
    letter = first.where(first.isin(["l", "n", "h"])).astype("object").fillna("")
    cls = cls.where(cls != "", letter)
    return cls.replace("", pd.NA).astype("string")


def parse_utc(acq_date, acq_time):
    """acq_date YYYY-MM-DD + acq_time HHMM (texto) -> timestamp UTC; invalidos -> NaT."""
    t = acq_time.astype("string").str.strip().str.zfill(4)
    hh = pd.to_numeric(t.str[:2], errors="coerce")
    mm = pd.to_numeric(t.str[2:], errors="coerce")
    ok = (t.str.len() == 4) & (hh < 24) & (mm < 60)
    ts = pd.to_datetime(acq_date.astype("string") + " " + t.str[:2] + ":" + t.str[2:],
                        format="%Y-%m-%d %H:%M", errors="coerce", utc=True)
    return t, ts.where(ok)


def harmonize(df, country, source_file):
    """Esquema comum MODIS+VIIRS. Mantem a proveniencia das bandas de brilho."""
    if df.empty:
        return None
    is_viirs = df["instrument"].astype(str).str.upper().eq("VIIRS").any() if "instrument" in df else False
    if is_viirs:
        if "bright_ti4" not in df and "brightness" in df:      # alguns servicos usam brightness/brightness_2
            df = df.rename(columns={"brightness": "bright_ti4", "brightness_2": "bright_ti5"})
        p_col, a_col = "bright_ti4", "bright_ti5"
    else:
        p_col, a_col = "brightness", "bright_t31"

    for c in NUMERIC + [p_col, a_col]:
        df[c] = pd.to_numeric(df[c], errors="coerce") if c in df else np.nan

    t, ts = parse_utc(df["acq_date"], df["acq_time"])
    out = pd.DataFrame({
        "country": country,
        "source_file": source_file,
        "instrument": df["instrument"],
        "satellite": df["satellite"],
        "version": df["version"].astype("string"),                       # texto: 6.03, 2, ...
        "latitude": df["latitude"],
        "longitude": df["longitude"],
        "acq_date": df["acq_date"],
        "acq_time": t,                                                   # HHMM com 4 digitos
        "acq_datetime_utc": ts,
        "daynight": df["daynight"],
        "type": pd.to_numeric(df["type"], errors="coerce").astype("Int8") if "type" in df else pd.array([pd.NA] * len(df), dtype="Int8"),
        "confidence_raw": df["confidence"].astype("string"),
        "confidence_class": confidence_class(df["confidence"]),
        "brightness_primary": df[p_col],
        "brightness_primary_band": p_col,
        "brightness_aux": df[a_col],
        "brightness_aux_band": a_col,
        "scan": df["scan"],
        "track": df["track"],
        "pixel_area_km2": df["scan"] * df["track"],
        "frp": df["frp"],
    })
    return out


def process_zip(sensor, year, zip_path, out_path):
    """Le todos os CSVs de paises de um zip e grava um Parquet. Devolve linha de resumo."""
    frames, n_csv = [], 0
    with tempfile.TemporaryDirectory() as tmp:
        local = Path(tmp) / zip_path.name
        shutil.copy(zip_path, local)                                     # disco local e mais rapido que a Drive
        with zipfile.ZipFile(local) as z:
            for info in z.infolist():
                m = CSV_RE.match(Path(info.filename).name)
                if info.is_dir() or not m:
                    continue
                n_csv += 1
                with z.open(info) as f:
                    df = pd.read_csv(f, dtype=str)                       # tudo texto; converte-se em harmonize()
                h = harmonize(df, m["country"], Path(info.filename).name)
                if h is not None:
                    frames.append(h)
    if not frames:
        return dict(sensor=sensor, year=year, n_csv=n_csv, n_rows=0, n_countries=0, n_bad_time=0, status="vazio")
    all_ = pd.concat(frames, ignore_index=True)
    tmp_out = out_path.with_suffix(".parquet.tmp")
    all_.to_parquet(tmp_out, index=False, compression="zstd")
    tmp_out.replace(out_path)                                            # so aparece completo
    return dict(sensor=sensor, year=year, n_csv=n_csv, n_rows=len(all_), n_countries=all_["country"].nunique(),
                n_bad_time=int(all_["acq_datetime_utc"].isna().sum()), status="ok")

In [ ]:
zips = find_zips(RAW)
todo = sorted(k for k in zips
              if (SENSORS is None or k[0] in SENSORS) and (YEARS is None or k[1] in YEARS))
print(f"{len(zips)} zips encontrados, {len(todo)} selecionados")

log = []
for sensor, year in todo:
    out = PROCESSED / f"{sensor}_{year}.parquet"
    if out.exists() and not FORCE:
        print(f"[salta] {out.name} ja existe")
        continue
    print(f"[processa] {sensor} {year} <- {zips[(sensor, year)].name}")
    row = process_zip(sensor, year, zips[(sensor, year)], out)
    print("   ", row)
    log.append(row)

if log:
    log_path = PROCESSED / "_ingest_log.csv"
    new = pd.DataFrame(log)
    if log_path.exists():
        new = pd.concat([pd.read_csv(log_path), new]).drop_duplicates(["sensor", "year"], keep="last")
    new.sort_values(["sensor", "year"]).to_csv(log_path, index=False)
    display(new.sort_values(["sensor", "year"]))

## Verificação rápida
Confirma o esquema e que `acq_time` tem 4 dígitos num dos Parquet gerados.

In [ ]:
files = sorted(PROCESSED.glob("*.parquet"))
print([f.name for f in files][:10], "...")
if files:
    s = pd.read_parquet(files[0])
    display(s.dtypes.to_frame("dtype").T)
    display(s.head(3))
    print("acq_time len:", s["acq_time"].str.len().value_counts().to_dict())
    print("confidence_class:", s["confidence_class"].value_counts(dropna=False).to_dict())
    print("type:", s["type"].value_counts(dropna=False).to_dict())